## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load and Explore Data

In [ ]:
from pathlib import Path


def _resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'Seminar QF', cwd.parent]

    for candidate in candidates:
        if (candidate / 'data' / 'output' / 'regime_switching_parameters.csv').exists():
            return candidate

    tried = '\n'.join(str(c / 'data' / 'output' / 'regime_switching_parameters.csv') for c in candidates)
    raise FileNotFoundError(f'Could not locate regime_switching_parameters.csv. Tried:\n{tried}')


PROJECT_ROOT = _resolve_project_root()
DATA_OUTPUT = PROJECT_ROOT / 'data' / 'output'
print(f'Using project root: {PROJECT_ROOT}')

rs_params = pd.read_csv(DATA_OUTPUT / 'regime_switching_parameters.csv')
ms_garch = pd.read_csv(DATA_OUTPUT / 'ms_garch_parameters.csv')

rs_params['date'] = pd.to_datetime(rs_params['date'])
ms_garch['date'] = pd.to_datetime(ms_garch['date'])

print('Regime Switching Parameters:')
print(f'Shape: {rs_params.shape}')
print(f'Date range: {rs_params["date"].min()} to {rs_params["date"].max()}')
print(f'Number of companies: {rs_params["gvkey"].nunique()}')
print()
print('MS-GARCH Parameters:')
print(f'Shape: {ms_garch.shape}')
print(f'Date range: {ms_garch["date"].min()} to {ms_garch["date"].max()}')
print(f'Number of companies: {ms_garch["gvkey"].nunique()}')

In [ ]:
print('Regime Switching Parameters - First 5 rows:')
display(rs_params.head())

print('\nMS-GARCH Parameters - First 5 rows:')
display(ms_garch.head())

In [ ]:
print('Regime Switching - Summary Statistics:')
display(rs_params.describe())

print('\nMS-GARCH - Summary Statistics:')
display(ms_garch.describe())

## Regime Frequency Analysis

In [ ]:
def calculate_steady_state_probabilities(p00, p11):
    pi_0 = (1 - p11) / (2 - p00 - p11)
    pi_1 = (1 - p00) / (2 - p00 - p11)
    return pi_0, pi_1

rs_params['steady_state_regime_0'] = rs_params.apply(
    lambda row: calculate_steady_state_probabilities(row['transition_prob_00'], row['transition_prob_11'])[0],
    axis=1
)
rs_params['steady_state_regime_1'] = rs_params.apply(
    lambda row: calculate_steady_state_probabilities(row['transition_prob_00'], row['transition_prob_11'])[1],
    axis=1
)

ms_garch['steady_state_regime_0'] = ms_garch.apply(
    lambda row: calculate_steady_state_probabilities(row['p00'], row['p11'])[0],
    axis=1
)
ms_garch['steady_state_regime_1'] = ms_garch.apply(
    lambda row: calculate_steady_state_probabilities(row['p00'], row['p11'])[1],
    axis=1
)

print('Regime Switching - Average Time in Each Regime:')
print(f'Regime 0: {rs_params["steady_state_regime_0"].mean():.2%}')
print(f'Regime 1: {rs_params["steady_state_regime_1"].mean():.2%}')

print('\nMS-GARCH - Average Time in Each Regime:')
print(f'Regime 0: {ms_garch["steady_state_regime_0"].mean():.2%}')
print(f'Regime 1: {ms_garch["steady_state_regime_1"].mean():.2%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rs_regime_freq = rs_params[['steady_state_regime_0', 'steady_state_regime_1']].mean()
axes[0].bar(['Regime 0', 'Regime 1'], rs_regime_freq, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0].set_ylabel('Probability')
axes[0].set_title('Regime Switching: Average Time in Each Regime')
axes[0].set_ylim([0, 1])
for i, v in enumerate(rs_regime_freq):
    axes[0].text(i, v + 0.02, f'{v:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ms_regime_freq = ms_garch[['steady_state_regime_0', 'steady_state_regime_1']].mean()
axes[1].bar(['Regime 0', 'Regime 1'], ms_regime_freq, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[1].set_ylabel('Probability')
axes[1].set_title('MS-GARCH: Average Time in Each Regime')
axes[1].set_ylim([0, 1])
for i, v in enumerate(ms_regime_freq):
    axes[1].text(i, v + 0.02, f'{v:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

rs_time = rs_params.groupby('date')[['steady_state_regime_0', 'steady_state_regime_1']].mean()
axes[0].plot(rs_time.index, rs_time['steady_state_regime_0'], label='Regime 0', color='#3498db', linewidth=2)
axes[0].plot(rs_time.index, rs_time['steady_state_regime_1'], label='Regime 1', color='#e74c3c', linewidth=2)
axes[0].fill_between(rs_time.index, 0, 1, alpha=0.1, color='gray')
axes[0].set_ylabel('Probability')
axes[0].set_title('Regime Switching: Regime Probabilities Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

ms_time = ms_garch.groupby('date')[['steady_state_regime_0', 'steady_state_regime_1']].mean()
axes[1].plot(ms_time.index, ms_time['steady_state_regime_0'], label='Regime 0', color='#3498db', linewidth=2)
axes[1].plot(ms_time.index, ms_time['steady_state_regime_1'], label='Regime 1', color='#e74c3c', linewidth=2)
axes[1].fill_between(ms_time.index, 0, 1, alpha=0.1, color='gray')
axes[1].set_ylabel('Probability')
axes[1].set_xlabel('Date')
axes[1].set_title('MS-GARCH: Regime Probabilities Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Regime Duration Analysis

In [ ]:
rs_params['duration_regime_0'] = 1 / (1 - rs_params['transition_prob_00'])
rs_params['duration_regime_1'] = 1 / (1 - rs_params['transition_prob_11'])

ms_garch['duration_regime_0'] = 1 / (1 - ms_garch['p00'])
ms_garch['duration_regime_1'] = 1 / (1 - ms_garch['p11'])

print('Regime Switching - Expected Duration (in periods):')
print(f'Regime 0: {rs_params["duration_regime_0"].mean():.2f} ± {rs_params["duration_regime_0"].std():.2f}')
print(f'Regime 1: {rs_params["duration_regime_1"].mean():.2f} ± {rs_params["duration_regime_1"].std():.2f}')

print('\nMS-GARCH - Expected Duration (in periods):')
print(f'Regime 0: {ms_garch["duration_regime_0"].mean():.2f} ± {ms_garch["duration_regime_0"].std():.2f}')
print(f'Regime 1: {ms_garch["duration_regime_1"].mean():.2f} ± {ms_garch["duration_regime_1"].std():.2f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def filter_duration(series, max_duration=1000):
    return series.replace([np.inf, -np.inf], np.nan).clip(upper=max_duration)

rs_dur_0_filtered = filter_duration(rs_params['duration_regime_0'])
axes[0, 0].hist(rs_dur_0_filtered.dropna(), bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(rs_dur_0_filtered.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].set_xlabel('Expected Duration (periods)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Regime Switching: Regime 0 Duration Distribution')
axes[0, 0].legend()

rs_dur_1_filtered = filter_duration(rs_params['duration_regime_1'])
axes[0, 1].hist(rs_dur_1_filtered.dropna(), bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(rs_dur_1_filtered.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 1].set_xlabel('Expected Duration (periods)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Regime Switching: Regime 1 Duration Distribution')
axes[0, 1].legend()

ms_dur_0_filtered = filter_duration(ms_garch['duration_regime_0'])
axes[1, 0].hist(ms_dur_0_filtered.dropna(), bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(ms_dur_0_filtered.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 0].set_xlabel('Expected Duration (periods)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('MS-GARCH: Regime 0 Duration Distribution')
axes[1, 0].legend()

ms_dur_1_filtered = filter_duration(ms_garch['duration_regime_1'])
axes[1, 1].hist(ms_dur_1_filtered.dropna(), bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(ms_dur_1_filtered.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].set_xlabel('Expected Duration (periods)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('MS-GARCH: Regime 1 Duration Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"\nDurations capped at 1000 periods for visualization")
print(f"Infinite/extreme durations filtered:")
print(f"  RS Regime 0: {rs_params['duration_regime_0'].isna().sum() + (rs_params['duration_regime_0'] > 1000).sum()}")
print(f"  RS Regime 1: {rs_params['duration_regime_1'].isna().sum() + (rs_params['duration_regime_1'] > 1000).sum()}")
print(f"  MS-GARCH Regime 0: {ms_garch['duration_regime_0'].isna().sum() + (ms_garch['duration_regime_0'] > 1000).sum()}")
print(f"  MS-GARCH Regime 1: {ms_garch['duration_regime_1'].isna().sum() + (ms_garch['duration_regime_1'] > 1000).sum()}")

## Parameter Comparison Between Regimes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rs_means = rs_params[['regime_0_mean', 'regime_1_mean']].mean() * 100
axes[0].bar(['Regime 0', 'Regime 1'], rs_means, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0].set_ylabel('Mean Return (%)')
axes[0].set_title('Regime Switching: Average Returns by Regime')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
for i, v in enumerate(rs_means):
    axes[0].text(i, v + (0.01 if v > 0 else -0.01), f'{v:.3f}%', ha='center', va='bottom' if v > 0 else 'top', fontsize=11)

ms_means = ms_garch[['mu_0', 'mu_1']].mean() * 100
axes[1].bar(['Regime 0', 'Regime 1'], ms_means, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[1].set_ylabel('Mean Return (%)')
axes[1].set_title('MS-GARCH: Average Returns by Regime')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
for i, v in enumerate(ms_means):
    axes[1].text(i, v + (0.01 if v > 0 else -0.01), f'{v:.3f}%', ha='center', va='bottom' if v > 0 else 'top', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print('='*80)
print('REGIME SWITCHING MODEL - PARAMETER COMPARISON')
print('='*80)
comparison_rs = pd.DataFrame({
    'Parameter': ['Mean Return', 'Volatility', 'Degrees of Freedom (nu)'],
    'Regime 0': [
        f"{rs_params['regime_0_mean'].mean()*100:.4f}%",
        f"{rs_params['regime_0_vol'].mean()*100:.4f}%",
        f"{rs_params['regime_0_nu'].mean():.2f}"
    ],
    'Regime 1': [
        f"{rs_params['regime_1_mean'].mean()*100:.4f}%",
        f"{rs_params['regime_1_vol'].mean()*100:.4f}%",
        f"{rs_params['regime_1_nu'].mean():.2f}"
    ],
    'Difference': [
        f"{(rs_params['regime_1_mean'].mean() - rs_params['regime_0_mean'].mean())*100:.4f}%",
        f"{(rs_params['regime_1_vol'].mean() - rs_params['regime_0_vol'].mean())*100:.4f}%",
        f"{rs_params['regime_1_nu'].mean() - rs_params['regime_0_nu'].mean():.2f}"
    ]
})
display(comparison_rs)

print('\n' + '='*80)
print('MS-GARCH MODEL - PARAMETER COMPARISON')
print('='*80)
comparison_ms = pd.DataFrame({
    'Parameter': ['Mean Return', 'Degrees of Freedom (nu)', 'Omega', 'Alpha', 'Beta'],
    'Regime 0': [
        f"{ms_garch['mu_0'].mean()*100:.4f}%",
        f"{ms_garch['nu_0'].mean():.2f}",
        f"{ms_garch['omega_0'].mean():.6f}",
        f"{ms_garch['alpha_0'].mean():.6f}",
        f"{ms_garch['beta_0'].mean():.6f}"
    ],
    'Regime 1': [
        f"{ms_garch['mu_1'].mean()*100:.4f}%",
        f"{ms_garch['nu_1'].mean():.2f}",
        f"{ms_garch['omega_1'].mean():.6f}",
        f"{ms_garch['alpha_1'].mean():.6f}",
        f"{ms_garch['beta_1'].mean():.6f}"
    ],
    'Ratio (R1/R0)': [
        f"{ms_garch['mu_1'].mean() / ms_garch['mu_0'].mean() if ms_garch['mu_0'].mean() != 0 else np.nan:.2f}",
        f"{ms_garch['nu_1'].mean() / ms_garch['nu_0'].mean():.2f}",
        f"{ms_garch['omega_1'].mean() / ms_garch['omega_0'].mean() if ms_garch['omega_0'].mean() != 0 else np.nan:.2f}",
        f"{ms_garch['alpha_1'].mean() / ms_garch['alpha_0'].mean() if ms_garch['alpha_0'].mean() != 0 else np.nan:.2f}",
        f"{ms_garch['beta_1'].mean() / ms_garch['beta_0'].mean():.2f}"
    ]
})
display(comparison_ms)

## Volatility Analysis by Regime

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rs_vols = rs_params[['regime_0_vol', 'regime_1_vol']].mean() * 100
axes[0].bar(['Regime 0', 'Regime 1'], rs_vols, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0].set_ylabel('Volatility (%)')
axes[0].set_title('Regime Switching: Average Volatility by Regime')
for i, v in enumerate(rs_vols):
    axes[0].text(i, v + 0.1, f'{v:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ms_garch['persist_0'] = ms_garch['alpha_0'] + ms_garch['beta_0']
ms_garch['persist_1'] = ms_garch['alpha_1'] + ms_garch['beta_1']

for r in [0, 1]:
    p_col = f'persist_{r}'
    denom = 1 - ms_garch[p_col]
    denom_safe = denom.where(denom > 0.001, other=np.nan)
    ms_garch[f'uncond_vol_{r}'] = np.sqrt(ms_garch[f'omega_{r}'] / denom_safe)

def trimmed_mean(s, pct=0.05):
    s = s.dropna()
    lo, hi = s.quantile(pct), s.quantile(1 - pct)
    return s[(s >= lo) & (s <= hi)].mean()

ms_vols_robust = pd.Series({
    'uncond_vol_0': trimmed_mean(ms_garch['uncond_vol_0']) * 100,
    'uncond_vol_1': trimmed_mean(ms_garch['uncond_vol_1']) * 100,
})

axes[1].bar(['Regime 0', 'Regime 1'], ms_vols_robust, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[1].set_ylabel('Unconditional Volatility (%)')
axes[1].set_title('MS-GARCH: Robust Unconditional Vol by Regime\n(trimmed mean, excl. near-IGARCH)')
for i, v in enumerate(ms_vols_robust):
    axes[1].text(i, v + 0.05, f'{v:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

ms_vols_naive = ms_garch[['uncond_vol_0', 'uncond_vol_1']].mean() * 100
print(f"Regime Switching:  Vol Ratio (R1/R0) = {rs_vols['regime_1_vol'] / rs_vols['regime_0_vol']:.2f}x")
print(f"MS-GARCH (naive):  Vol Ratio (R1/R0) = {ms_vols_naive['uncond_vol_1'] / ms_vols_naive['uncond_vol_0']:.2f}x  ⚠️ inflated by near-IGARCH rows")
print(f"MS-GARCH (robust): Vol Ratio (R1/R0) = {ms_vols_robust['uncond_vol_1'] / ms_vols_robust['uncond_vol_0']:.2f}x")
print()
n_total = len(ms_garch)
for r in [0, 1]:
    n_igarch = (ms_garch[f'persist_{r}'] >= 0.999).sum()
    n_nan = ms_garch[f'uncond_vol_{r}'].isna().sum()
    print(f"Regime {r}: {n_igarch} of {n_total} rows ({n_igarch/n_total*100:.1f}%) "
          f"have persistence ≥ 0.999 → uncond vol set to NaN")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].hist(rs_params['regime_0_vol']*100, bins=50, alpha=0.5, label='Regime 0', color='#3498db', edgecolor='black')
axes[0].hist(rs_params['regime_1_vol']*100, bins=50, alpha=0.5, label='Regime 1', color='#e74c3c', edgecolor='black')
axes[0].axvline(rs_params['regime_0_vol'].mean()*100, color='#3498db', linestyle='--', linewidth=2)
axes[0].axvline(rs_params['regime_1_vol'].mean()*100, color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_xlabel('Volatility (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Regime Switching: Volatility Distribution by Regime')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

v0 = ms_garch['uncond_vol_0'].dropna() * 100
v1 = ms_garch['uncond_vol_1'].dropna() * 100
cap = max(v0.quantile(0.99), v1.quantile(0.99))
axes[1].hist(v0[v0 <= cap], bins=50, alpha=0.5, label='Regime 0', color='#3498db', edgecolor='black')
axes[1].hist(v1[v1 <= cap], bins=50, alpha=0.5, label='Regime 1', color='#e74c3c', edgecolor='black')
axes[1].axvline(trimmed_mean(ms_garch['uncond_vol_0'])*100, color='#3498db', linestyle='--', linewidth=2)
axes[1].axvline(trimmed_mean(ms_garch['uncond_vol_1'])*100, color='#e74c3c', linestyle='--', linewidth=2)
axes[1].set_xlabel('Unconditional Volatility (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('MS-GARCH: Unconditional Vol Distribution (capped at P99, dashed = trimmed mean)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
ms_garch['persistence_0'] = ms_garch['alpha_0'] + ms_garch['beta_0']
ms_garch['persistence_1'] = ms_garch['alpha_1'] + ms_garch['beta_1']
ms_garch['uncond_var_0']  = ms_garch['omega_0'] / (1 - ms_garch['persistence_0'])
ms_garch['uncond_var_1']  = ms_garch['omega_1'] / (1 - ms_garch['persistence_1'])
ms_garch['uncond_vol_0']  = np.sqrt(ms_garch['uncond_var_0'].clip(lower=0)) * 100
ms_garch['uncond_vol_1']  = np.sqrt(ms_garch['uncond_var_1'].clip(lower=0)) * 100

THRESHOLD = 10
extreme = ms_garch[
    (ms_garch['uncond_vol_0'] > THRESHOLD) | (ms_garch['uncond_vol_1'] > THRESHOLD)
].copy()

print(f"Rows with unconditional vol > {THRESHOLD}%: {len(extreme)} / {len(ms_garch)}")
print(f"Unique firms involved: {extreme['gvkey'].nunique()}\n")

extreme_summary = (
    extreme
    .groupby('gvkey')
    .agg(
        n_extreme   = ('date', 'size'),
        date_min    = ('date', 'min'),
        date_max    = ('date', 'max'),
        max_vol_0   = ('uncond_vol_0', 'max'),
        max_vol_1   = ('uncond_vol_1', 'max'),
        median_p0   = ('persistence_0', 'median'),
        median_p1   = ('persistence_1', 'median'),
    )
    .sort_values('max_vol_1', ascending=False)
)
print("Per-firm summary of extreme unconditional volatilities:")
display(extreme_summary)

print("\nTop 20 most extreme individual (firm × date) observations:")
top_extreme = (
    extreme
    .assign(max_uncond_vol=lambda d: d[['uncond_vol_0','uncond_vol_1']].max(axis=1))
    .nlargest(20, 'max_uncond_vol')
    [['gvkey','date','omega_0','alpha_0','beta_0','persistence_0','uncond_vol_0',
      'omega_1','alpha_1','beta_1','persistence_1','uncond_vol_1']]
)
display(top_extreme)

extreme_by_date = extreme.groupby('date').size()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(extreme_by_date.index, extreme_by_date.values, width=25, color='crimson', alpha=0.7)
axes[0].set_title(f'Number of firms with uncond vol > {THRESHOLD}% per date')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Count')

axes[1].scatter(ms_garch['persistence_1'], ms_garch['uncond_vol_1'],
                alpha=0.15, s=8, color='steelblue', label='All')
axes[1].scatter(extreme['persistence_1'], extreme['uncond_vol_1'],
                alpha=0.5, s=12, color='red', label=f'Vol > {THRESHOLD}%')
axes[1].set_xlabel('Persistence (α₁ + β₁)')
axes[1].set_ylabel('Unconditional Vol Regime 1 (%)')
axes[1].set_title('Persistence vs Unconditional Volatility (Regime 1)')
axes[1].legend()
axes[1].set_ylim(0, ms_garch['uncond_vol_1'].quantile(0.99)*1.1)

plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0.999, 0.998, 0.997, 0.995, 0.993, 0.99, 0.985, 0.98, 0.975, 0.97, 0.96, 0.95]

p0 = ms_garch['alpha_0'] + ms_garch['beta_0']
p1 = ms_garch['alpha_1'] + ms_garch['beta_1']
max_p = np.maximum(p0, p1)

rows = []
for thr in thresholds:
    mask = max_p < thr
    kept = mask.sum()
    pct_kept = kept / len(ms_garch) * 100
    
    uv0 = np.sqrt((ms_garch.loc[mask, 'omega_0'] / (1 - p0[mask])).clip(lower=0)) * 100
    uv1 = np.sqrt((ms_garch.loc[mask, 'omega_1'] / (1 - p1[mask])).clip(lower=0)) * 100
    
    rows.append({
        'persistence_cap': thr,
        'rows_kept': kept,
        'rows_excluded': len(ms_garch) - kept,
        'pct_kept': f'{pct_kept:.1f}%',
        'max_uncond_vol_R0 (%)': f'{uv0.max():.1f}',
        'max_uncond_vol_R1 (%)': f'{uv1.max():.1f}',
        'p99_uncond_vol_R1 (%)': f'{uv1.quantile(0.99):.1f}',
        'p95_uncond_vol_R1 (%)': f'{uv1.quantile(0.95):.1f}',
        'median_uncond_vol_R1 (%)': f'{uv1.median():.2f}',
    })

cap_analysis = pd.DataFrame(rows)
print("Effect of different persistence caps on MS-GARCH unconditional volatility:\n")
display(cap_analysis)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(max_p, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0.99, color='red', ls='--', lw=2, label='0.99')
axes[0].axvline(0.995, color='orange', ls='--', lw=2, label='0.995')
axes[0].axvline(0.998, color='green', ls='--', lw=2, label='0.998')
axes[0].set_xlabel('Max persistence (α + β)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of max(persistence₀, persistence₁)')
axes[0].legend()

tail = max_p[max_p > 0.95]
axes[1].hist(tail, bins=50, color='salmon', alpha=0.7, edgecolor='white')
axes[1].axvline(0.99, color='red', ls='--', lw=2, label='0.99')
axes[1].axvline(0.995, color='orange', ls='--', lw=2, label='0.995')
axes[1].axvline(0.998, color='green', ls='--', lw=2, label='0.998')
axes[1].set_xlabel('Max persistence (α + β)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Tail zoom: persistence > 0.95')
axes[1].legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
candidate_caps = [0.998, 0.995, 0.99, 0.98]

for ax, cap_val in zip(axes.flat, candidate_caps):
    mask = max_p < cap_val
    uv0 = np.sqrt((ms_garch.loc[mask, 'omega_0'] / (1 - p0[mask])).clip(lower=0)) * 100
    uv1 = np.sqrt((ms_garch.loc[mask, 'omega_1'] / (1 - p1[mask])).clip(lower=0)) * 100
    
    ax.hist(uv0, bins=50, alpha=0.5, color='steelblue', label='Regime 0')
    ax.hist(uv1, bins=50, alpha=0.5, color='salmon', label='Regime 1')
    ax.axvline(uv0.median(), color='blue', ls='--', lw=1.5)
    ax.axvline(uv1.median(), color='red', ls='--', lw=1.5)
    excluded_pct = (1 - mask.sum()/len(ms_garch)) * 100
    ax.set_title(f'Cap = {cap_val} (excl. {excluded_pct:.1f}% of rows)\n'
                 f'Max R1 vol = {uv1.max():.1f}%, Median R1 = {uv1.median():.2f}%')
    ax.set_xlabel('Unconditional Volatility (%)')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('MS-GARCH Unconditional Vol Distribution under Different Persistence Caps', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Transition Probability Analysis

In [ ]:
print('='*80)
print('REGIME SWITCHING - TRANSITION PROBABILITIES')
print('='*80)
print(f"P(stay in Regime 0 | in Regime 0): {rs_params['transition_prob_00'].mean():.4f}")
print(f"P(switch to Regime 1 | in Regime 0): {rs_params['transition_prob_01'].mean():.4f}")
print(f"P(switch to Regime 0 | in Regime 1): {rs_params['transition_prob_10'].mean():.4f}")
print(f"P(stay in Regime 1 | in Regime 1): {rs_params['transition_prob_11'].mean():.4f}")

print('\n' + '='*80)
print('MS-GARCH - TRANSITION PROBABILITIES')
print('='*80)
print(f"P(stay in Regime 0 | in Regime 0): {ms_garch['p00'].mean():.4f}")
print(f"P(switch to Regime 1 | in Regime 0): {(1 - ms_garch['p00']).mean():.4f}")
print(f"P(switch to Regime 0 | in Regime 1): {(1 - ms_garch['p11']).mean():.4f}")
print(f"P(stay in Regime 1 | in Regime 1): {ms_garch['p11'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(rs_params['transition_prob_00'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(rs_params['transition_prob_00'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].set_xlabel('Probability')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Regime Switching: P(Stay in Regime 0)')
axes[0, 0].legend()

axes[0, 1].hist(rs_params['transition_prob_11'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(rs_params['transition_prob_11'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 1].set_xlabel('Probability')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Regime Switching: P(Stay in Regime 1)')
axes[0, 1].legend()

axes[1, 0].hist(ms_garch['p00'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(ms_garch['p00'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 0].set_xlabel('Probability')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('MS-GARCH: P(Stay in Regime 0)')
axes[1, 0].legend()

axes[1, 1].hist(ms_garch['p11'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(ms_garch['p11'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].set_xlabel('Probability')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('MS-GARCH: P(Stay in Regime 1)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

rs_trans_time = rs_params.groupby('date')[['transition_prob_00', 'transition_prob_11']].mean()
axes[0].plot(rs_trans_time.index, rs_trans_time['transition_prob_00'], label='P(Stay in R0)', color='#3498db', linewidth=2)
axes[0].plot(rs_trans_time.index, rs_trans_time['transition_prob_11'], label='P(Stay in R1)', color='#e74c3c', linewidth=2)
axes[0].set_ylabel('Probability')
axes[0].set_title('Regime Switching: Regime Persistence Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

ms_trans_time = ms_garch.groupby('date')[['p00', 'p11']].mean()
axes[1].plot(ms_trans_time.index, ms_trans_time['p00'], label='P(Stay in R0)', color='#3498db', linewidth=2)
axes[1].plot(ms_trans_time.index, ms_trans_time['p11'], label='P(Stay in R1)', color='#e74c3c', linewidth=2)
axes[1].set_ylabel('Probability')
axes[1].set_xlabel('Date')
axes[1].set_title('MS-GARCH: Regime Persistence Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Time Series Visualization of Regimes

In [ ]:
sample_gvkey = rs_params['gvkey'].iloc[0]
print(f"Analyzing company: {sample_gvkey}")

rs_company = rs_params[rs_params['gvkey'] == sample_gvkey].sort_values('date')
ms_company = ms_garch[ms_garch['gvkey'] == sample_gvkey].sort_values('date')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

axes[0].plot(rs_company['date'], rs_company['steady_state_regime_0'], label='Regime 0', color='#3498db', linewidth=2)
axes[0].plot(rs_company['date'], rs_company['steady_state_regime_1'], label='Regime 1', color='#e74c3c', linewidth=2)
axes[0].fill_between(rs_company['date'], 0, 1, alpha=0.1, color='gray')
axes[0].set_ylabel('Probability')
axes[0].set_title(f'Company {sample_gvkey}: Regime Probabilities Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(rs_company['date'], rs_company['regime_0_mean']*100, label='Regime 0 Mean', color='#3498db', linewidth=2)
axes[1].plot(rs_company['date'], rs_company['regime_1_mean']*100, label='Regime 1 Mean', color='#e74c3c', linewidth=2)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('Mean Return (%)')
axes[1].set_title('Mean Returns by Regime')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(rs_company['date'], rs_company['regime_0_vol']*100, label='Regime 0 Volatility', color='#3498db', linewidth=2)
axes[2].plot(rs_company['date'], rs_company['regime_1_vol']*100, label='Regime 1 Volatility', color='#e74c3c', linewidth=2)
axes[2].set_ylabel('Volatility (%)')
axes[2].set_title('Volatility by Regime')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(rs_company['date'], rs_company['transition_prob_00'], label='P(Stay in R0)', color='#3498db', linewidth=2)
axes[3].plot(rs_company['date'], rs_company['transition_prob_11'], label='P(Stay in R1)', color='#e74c3c', linewidth=2)
axes[3].set_ylabel('Probability')
axes[3].set_xlabel('Date')
axes[3].set_title('Transition Probabilities')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Statistical Tests for Regime Differences

In [ ]:
from scipy.stats import ttest_rel, ttest_ind

print('='*80)
print('REGIME SWITCHING - STATISTICAL TESTS FOR PARAMETER DIFFERENCES')
print('='*80)

t_stat, p_value = ttest_rel(rs_params['regime_0_mean'], rs_params['regime_1_mean'])
print(f"\nMean Returns:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

t_stat, p_value = ttest_rel(rs_params['regime_0_vol'], rs_params['regime_1_vol'])
print(f"\nVolatility:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

t_stat, p_value = ttest_rel(rs_params['regime_0_nu'], rs_params['regime_1_nu'])
print(f"\nDegrees of Freedom (nu):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

print('\n' + '='*80)
print('MS-GARCH - STATISTICAL TESTS FOR PARAMETER DIFFERENCES')
print('='*80)

t_stat, p_value = ttest_rel(ms_garch['mu_0'], ms_garch['mu_1'])
print(f"\nMean Returns:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

t_stat, p_value = ttest_rel(ms_garch['alpha_0'], ms_garch['alpha_1'])
print(f"\nAlpha (ARCH effect):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

t_stat, p_value = ttest_rel(ms_garch['beta_0'], ms_garch['beta_1'])
print(f"\nBeta (GARCH effect):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at 5% level: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
def cohens_d(x, y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)

print('='*80)
print('EFFECT SIZES (Cohen\'s d)')
print('='*80)
print("\nRegime Switching:")
print(f"  Mean Returns: d = {cohens_d(rs_params['regime_0_mean'], rs_params['regime_1_mean']):.4f}")
print(f"  Volatility: d = {cohens_d(rs_params['regime_0_vol'], rs_params['regime_1_vol']):.4f}")
print(f"  Degrees of Freedom: d = {cohens_d(rs_params['regime_0_nu'], rs_params['regime_1_nu']):.4f}")

print("\nMS-GARCH:")
print(f"  Mean Returns: d = {cohens_d(ms_garch['mu_0'], ms_garch['mu_1']):.4f}")
print(f"  Alpha: d = {cohens_d(ms_garch['alpha_0'], ms_garch['alpha_1']):.4f}")
print(f"  Beta: d = {cohens_d(ms_garch['beta_0'], ms_garch['beta_1']):.4f}")

## Cross-Sectional Analysis

In [ ]:
company_stats_rs = rs_params.groupby('gvkey').agg({
    'regime_0_mean': 'mean',
    'regime_1_mean': 'mean',
    'regime_0_vol': 'mean',
    'regime_1_vol': 'mean',
    'steady_state_regime_0': 'mean',
    'steady_state_regime_1': 'mean',
    'duration_regime_0': 'mean',
    'duration_regime_1': 'mean'
}).reset_index()

company_stats_rs['vol_ratio'] = company_stats_rs['regime_1_vol'] / company_stats_rs['regime_0_vol']

print('Company-Level Statistics (Regime Switching):')
display(company_stats_rs.head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(company_stats_rs['vol_ratio'], bins=30, color='purple', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(company_stats_rs['vol_ratio'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].set_xlabel('Volatility Ratio (R1/R0)')
axes[0, 0].set_ylabel('Number of Companies')
axes[0, 0].set_title('Distribution of Volatility Ratios Across Companies')
axes[0, 0].legend()

axes[0, 1].scatter(company_stats_rs['duration_regime_0'], company_stats_rs['duration_regime_1'], alpha=0.6, color='green')
axes[0, 1].plot([0, company_stats_rs['duration_regime_0'].max()], [0, company_stats_rs['duration_regime_0'].max()], 'r--', label='Equal Duration')
axes[0, 1].set_xlabel('Regime 0 Duration')
axes[0, 1].set_ylabel('Regime 1 Duration')
axes[0, 1].set_title('Regime Durations: R0 vs R1')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(company_stats_rs['steady_state_regime_0'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(company_stats_rs['steady_state_regime_0'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 0].set_xlabel('Probability')
axes[1, 0].set_ylabel('Number of Companies')
axes[1, 0].set_title('Distribution of Time Spent in Regime 0')
axes[1, 0].legend()

company_stats_rs['mean_diff'] = company_stats_rs['regime_1_mean'] - company_stats_rs['regime_0_mean']
axes[1, 1].hist(company_stats_rs['mean_diff']*100, bins=30, color='orange', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(company_stats_rs['mean_diff'].mean()*100, color='red', linestyle='--', linewidth=2, label='Mean')
axes[1, 1].axvline(0, color='black', linestyle='-', linewidth=1)
axes[1, 1].set_xlabel('Mean Return Difference (R1 - R0) %')
axes[1, 1].set_ylabel('Number of Companies')
axes[1, 1].set_title('Distribution of Mean Return Differences')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('='*80)
print('KEY FINDINGS SUMMARY')
print('='*80)

print('\n1. REGIME FREQUENCY:')
print(f"   - Average time in Regime 0 (RS): {rs_params['steady_state_regime_0'].mean():.2%}")
print(f"   - Average time in Regime 1 (RS): {rs_params['steady_state_regime_1'].mean():.2%}")
print(f"   - Average time in Regime 0 (MS-GARCH): {ms_garch['steady_state_regime_0'].mean():.2%}")
print(f"   - Average time in Regime 1 (MS-GARCH): {ms_garch['steady_state_regime_1'].mean():.2%}")

print('\n2. REGIME PERSISTENCE:')
print(f"   - Regime 0 expected duration (RS): {rs_params['duration_regime_0'].mean():.2f} periods")
print(f"   - Regime 1 expected duration (RS): {rs_params['duration_regime_1'].mean():.2f} periods")
print(f"   - Regime 0 expected duration (MS-GARCH): {ms_garch['duration_regime_0'].mean():.2f} periods")
print(f"   - Regime 1 expected duration (MS-GARCH): {ms_garch['duration_regime_1'].mean():.2f} periods")

print('\n3. PARAMETER DIFFERENCES:')
print(f"   - Mean return difference (RS): {(rs_params['regime_1_mean'].mean() - rs_params['regime_0_mean'].mean())*100:.4f}%")
print(f"   - Volatility ratio (RS): {(rs_params['regime_1_vol'].mean() / rs_params['regime_0_vol'].mean()):.2f}x")
print(f"   - Mean return difference (MS-GARCH): {(ms_garch['mu_1'].mean() - ms_garch['mu_0'].mean())*100:.4f}%")

print('\n4. CROSS-SECTIONAL VARIATION:')
print(f"   - Volatility ratio std dev: {company_stats_rs['vol_ratio'].std():.2f}")
print(f"   - Companies with higher vol in R1: {(company_stats_rs['vol_ratio'] > 1).sum()} / {len(company_stats_rs)}")
print(f"   - Companies with positive mean diff: {(company_stats_rs['mean_diff'] > 0).sum()} / {len(company_stats_rs)}")

print('\n' + '='*80)

## Leverage Group Analysis

In [ ]:
if 'DATA_OUTPUT' in globals():
    merged_path = DATA_OUTPUT / 'merged_data_with_merton.csv'
elif 'PROJECT_ROOT' in globals():
    merged_path = PROJECT_ROOT / 'data' / 'output' / 'merged_data_with_merton.csv'
else:
    from pathlib import Path
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'Seminar QF', cwd.parent]
    merged_path = None
    for candidate in candidates:
        candidate_path = candidate / 'data' / 'output' / 'merged_data_with_merton.csv'
        if candidate_path.exists():
            merged_path = candidate_path
            break
    if merged_path is None:
        tried = '\n'.join(str(c / 'data' / 'output' / 'merged_data_with_merton.csv') for c in candidates)
        raise FileNotFoundError(f'Could not locate merged_data_with_merton.csv. Tried:\n{tried}')

print(f'Using merged data file: {merged_path}')
df_merged = pd.read_csv(merged_path)
df_merged['date'] = pd.to_datetime(df_merged['date'])

df_merged['leverage_ratio'] = df_merged['liabilities_total'] / (df_merged['liabilities_total'] + df_merged['mkt_cap'])

firm_leverage = df_merged.groupby('gvkey').agg({
    'leverage_ratio': 'mean',
    'company': 'first'
}).reset_index()

firm_leverage.columns = ['gvkey', 'avg_leverage', 'company']

firm_leverage['leverage_tercile'] = pd.qcut(
    firm_leverage['avg_leverage'], 
    q=3, 
    labels=['Low Leverage', 'Mid Leverage', 'High Leverage']
)

tercile_boundaries = pd.qcut(firm_leverage['avg_leverage'], q=3, retbins=True)[1]

print("Leverage Group Definitions:")
print(f"Low Leverage:  < {tercile_boundaries[1]:.2%}")
print(f"Mid Leverage:  {tercile_boundaries[1]:.2%} - {tercile_boundaries[2]:.2%}")
print(f"High Leverage: > {tercile_boundaries[2]:.2%}")

leverage_group_map = firm_leverage.set_index('gvkey')['leverage_tercile'].to_dict()

rs_params['leverage_tercile'] = rs_params['gvkey'].map(leverage_group_map)
ms_garch['leverage_tercile'] = ms_garch['gvkey'].map(leverage_group_map)

print(f"\nLeverage groups assigned")
print(f"  Low Leverage: {(rs_params['leverage_tercile'] == 'Low Leverage').sum()} observations")
print(f"  Mid Leverage: {(rs_params['leverage_tercile'] == 'Mid Leverage').sum()} observations")
print(f"  High Leverage: {(rs_params['leverage_tercile'] == 'High Leverage').sum()} observations")

In [ ]:
print('='*80)
print('REGIME FREQUENCY BY LEVERAGE GROUP')
print('='*80)

for model_name, df, col_prefix in [('Regime Switching', rs_params, 'steady_state_regime_'), 
                                     ('MS-GARCH', ms_garch, 'steady_state_regime_')]:
    print(f"\n{model_name}:")
    print(f"{'-'*80}")
    
    for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
        group_data = df[df['leverage_tercile'] == leverage_group]
        regime_0_prob = group_data[f'{col_prefix}0'].mean()
        regime_1_prob = group_data[f'{col_prefix}1'].mean()
        
        print(f"\n  {leverage_group}:")
        print(f"    Regime 0: {regime_0_prob:.2%}")
        print(f"    Regime 1: {regime_1_prob:.2%}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

leverage_groups = ['Low Leverage', 'Mid Leverage', 'High Leverage']
colors_r0 = ['#2ecc71', '#f39c12', '#e74c3c']
colors_r1 = ['#27ae60', '#e67e22', '#c0392b']

for idx, leverage_group in enumerate(leverage_groups):
    ax = axes[0, idx]
    group_data = rs_params[rs_params['leverage_tercile'] == leverage_group]
    
    regime_probs = [
        group_data['steady_state_regime_0'].mean(),
        group_data['steady_state_regime_1'].mean()
    ]
    
    bars = ax.bar(['Regime 0', 'Regime 1'], regime_probs, 
                   color=[colors_r0[idx], colors_r1[idx]], alpha=0.7, edgecolor='black')
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title(f'Regime Switching: {leverage_group}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])

for idx, leverage_group in enumerate(leverage_groups):
    ax = axes[1, idx]
    group_data = ms_garch[ms_garch['leverage_tercile'] == leverage_group]
    
    regime_probs = [
        group_data['steady_state_regime_0'].mean(),
        group_data['steady_state_regime_1'].mean()
    ]
    
    bars = ax.bar(['Regime 0', 'Regime 1'], regime_probs, 
                   color=[colors_r0[idx], colors_r1[idx]], alpha=0.7, edgecolor='black')
    ax.set_ylabel('Probability', fontsize=11)
    ax.set_title(f'MS-GARCH: {leverage_group}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
print('='*80)
print('REGIME SWITCHING: PARAMETER COMPARISON BY LEVERAGE GROUP')
print('='*80)

rs_leverage_summary = []

for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
    group_data = rs_params[rs_params['leverage_tercile'] == leverage_group]
    
    rs_leverage_summary.append({
        'Leverage Group': leverage_group,
        'Mean R0 (%)': group_data['regime_0_mean'].mean() * 100,
        'Mean R1 (%)': group_data['regime_1_mean'].mean() * 100,
        'Vol R0 (%)': group_data['regime_0_vol'].mean() * 100,
        'Vol R1 (%)': group_data['regime_1_vol'].mean() * 100,
        'Vol Ratio': group_data['regime_1_vol'].mean() / group_data['regime_0_vol'].mean(),
        'Duration R0': group_data['duration_regime_0'].mean(),
        'Duration R1': group_data['duration_regime_1'].mean(),
        'P(R0)': group_data['steady_state_regime_0'].mean(),
        'P(R1)': group_data['steady_state_regime_1'].mean()
    })

rs_leverage_df = pd.DataFrame(rs_leverage_summary)
display(rs_leverage_df.round(4))

In [ ]:
print('='*80)
print('MS-GARCH: PARAMETER COMPARISON BY LEVERAGE GROUP')
print('='*80)

ms_leverage_summary = []

for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
    group_data = ms_garch[ms_garch['leverage_tercile'] == leverage_group]
    n = len(group_data)
    
    ms_leverage_summary.append({
        'Leverage Group': leverage_group,
        'Mean R0 (%)': group_data['mu_0'].mean() * 100,
        'Mean R1 (%)': group_data['mu_1'].mean() * 100,
        'Omega R0': group_data['omega_0'].mean(),
        'Omega R1': group_data['omega_1'].mean(),
        'Alpha R0': group_data['alpha_0'].mean(),
        'Alpha R1': group_data['alpha_1'].mean(),
        'Beta R0': group_data['beta_0'].mean(),
        'Beta R1': group_data['beta_1'].mean(),
        'Persist R0': group_data['persist_0'].mean(),
        'Persist R1': group_data['persist_1'].mean(),
        'Uncond Vol R0 (%)': trimmed_mean(group_data['uncond_vol_0']) * 100,
        'Uncond Vol R1 (%)': trimmed_mean(group_data['uncond_vol_1']) * 100,
        '% near-IGARCH R1': (group_data['persist_1'] >= 0.999).sum() / n * 100,
        'Duration R0': group_data['duration_regime_0'].mean(),
        'Duration R1': group_data['duration_regime_1'].mean(),
        'P(R0)': group_data['steady_state_regime_0'].mean(),
        'P(R1)': group_data['steady_state_regime_1'].mean()
    })

ms_leverage_df = pd.DataFrame(ms_leverage_summary)
display(ms_leverage_df.round(6))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

leverage_groups = ['Low Leverage', 'Mid Leverage', 'High Leverage']
colors = ['#2ecc71', '#f39c12', '#e74c3c']

ax = axes[0]
x = np.arange(len(leverage_groups))
width = 0.35

r0_vols = [rs_params[rs_params['leverage_tercile'] == lg]['regime_0_vol'].mean() * 100 
           for lg in leverage_groups]
r1_vols = [rs_params[rs_params['leverage_tercile'] == lg]['regime_1_vol'].mean() * 100 
           for lg in leverage_groups]

bars1 = ax.bar(x - width/2, r0_vols, width, label='Regime 0', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, r1_vols, width, label='Regime 1', color='#e74c3c', alpha=0.7, edgecolor='black')

ax.set_ylabel('Volatility (%)', fontsize=12)
ax.set_title('Regime Switching: Volatility by Leverage Group', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(leverage_groups)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=9)

ax = axes[1]
r0_vols_ms = [ms_garch[ms_garch['leverage_tercile'] == lg]['uncond_vol_0'].mean() * 100 
              for lg in leverage_groups]
r1_vols_ms = [ms_garch[ms_garch['leverage_tercile'] == lg]['uncond_vol_1'].mean() * 100 
              for lg in leverage_groups]

bars1 = ax.bar(x - width/2, r0_vols_ms, width, label='Regime 0', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, r1_vols_ms, width, label='Regime 1', color='#e74c3c', alpha=0.7, edgecolor='black')

ax.set_ylabel('Unconditional Volatility (%)', fontsize=12)
ax.set_title('MS-GARCH: Volatility by Leverage Group', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(leverage_groups)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

leverage_groups = ['Low Leverage', 'Mid Leverage', 'High Leverage']
x = np.arange(len(leverage_groups))
width = 0.35

ax = axes[0]
r0_durations = [rs_params[rs_params['leverage_tercile'] == lg]['duration_regime_0'].mean() 
                for lg in leverage_groups]
r1_durations = [rs_params[rs_params['leverage_tercile'] == lg]['duration_regime_1'].mean() 
                for lg in leverage_groups]

bars1 = ax.bar(x - width/2, r0_durations, width, label='Regime 0', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, r1_durations, width, label='Regime 1', color='#e74c3c', alpha=0.7, edgecolor='black')

ax.set_ylabel('Expected Duration (periods)', fontsize=12)
ax.set_title('Regime Switching: Regime Duration by Leverage Group', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(leverage_groups)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)

ax = axes[1]
r0_durations_ms = [ms_garch[ms_garch['leverage_tercile'] == lg]['duration_regime_0'].mean() 
                   for lg in leverage_groups]
r1_durations_ms = [ms_garch[ms_garch['leverage_tercile'] == lg]['duration_regime_1'].mean() 
                   for lg in leverage_groups]

bars1 = ax.bar(x - width/2, r0_durations_ms, width, label='Regime 0', color='#3498db', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, r1_durations_ms, width, label='Regime 1', color='#e74c3c', alpha=0.7, edgecolor='black')

ax.set_ylabel('Expected Duration (periods)', fontsize=12)
ax.set_title('MS-GARCH: Regime Duration by Leverage Group', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(leverage_groups)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

leverage_groups = ['Low Leverage', 'Mid Leverage', 'High Leverage']
colors = ['#2ecc71', '#f39c12', '#e74c3c']

for idx, leverage_group in enumerate(leverage_groups):
    ax = axes[idx]
    
    group_data = rs_params[rs_params['leverage_tercile'] == leverage_group]
    vol_ratios = group_data['regime_1_vol'] / group_data['regime_0_vol']
    
    ax.hist(vol_ratios, bins=30, color=colors[idx], alpha=0.7, edgecolor='black')
    ax.axvline(vol_ratios.mean(), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {vol_ratios.mean():.2f}')
    ax.axvline(vol_ratios.median(), color='blue', linestyle='--', linewidth=2, 
               label=f'Median: {vol_ratios.median():.2f}')
    
    ax.set_xlabel('Volatility Ratio (R1/R0)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{leverage_group}\n(Regime Switching)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Volatility Ratio Summary by Leverage Group (Regime Switching):')
print('='*80)
for leverage_group in leverage_groups:
    group_data = rs_params[rs_params['leverage_tercile'] == leverage_group]
    vol_ratios = group_data['regime_1_vol'] / group_data['regime_0_vol']
    
    print(f"\n{leverage_group}:")
    print(f"  Mean:   {vol_ratios.mean():.4f}")
    print(f"  Median: {vol_ratios.median():.4f}")
    print(f"  Std:    {vol_ratios.std():.4f}")
    print(f"  Min:    {vol_ratios.min():.4f}")
    print(f"  Max:    {vol_ratios.max():.4f}")

In [ ]:
print('='*80)
print('LEVERAGE GROUP ANALYSIS - KEY FINDINGS')
print('='*80)

print('\n1. REGIME FREQUENCY BY LEVERAGE:')
for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
    rs_group = rs_params[rs_params['leverage_tercile'] == leverage_group]
    ms_group = ms_garch[ms_garch['leverage_tercile'] == leverage_group]
    
    print(f"\n  {leverage_group}:")
    print(f"    RS - Regime 0: {rs_group['steady_state_regime_0'].mean():.2%}, Regime 1: {rs_group['steady_state_regime_1'].mean():.2%}")
    print(f"    MS - Regime 0: {ms_group['steady_state_regime_0'].mean():.2%}, Regime 1: {ms_group['steady_state_regime_1'].mean():.2%}")

print('\n2. VOLATILITY PATTERNS:')
for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
    rs_group = rs_params[rs_params['leverage_tercile'] == leverage_group]
    vol_ratio = rs_group['regime_1_vol'].mean() / rs_group['regime_0_vol'].mean()
    
    print(f"\n  {leverage_group}:")
    print(f"    Regime 0 Vol: {rs_group['regime_0_vol'].mean()*100:.2f}%")
    print(f"    Regime 1 Vol: {rs_group['regime_1_vol'].mean()*100:.2f}%")
    print(f"    Volatility Ratio: {vol_ratio:.2f}x")

print('\n3. REGIME PERSISTENCE:')
for leverage_group in ['Low Leverage', 'Mid Leverage', 'High Leverage']:
    rs_group = rs_params[rs_params['leverage_tercile'] == leverage_group]
    
    print(f"\n  {leverage_group}:")
    print(f"    Regime 0 Duration: {rs_group['duration_regime_0'].mean():.1f} periods")
    print(f"    Regime 1 Duration: {rs_group['duration_regime_1'].mean():.1f} periods")

print('\n' + '='*80)

## MS-GARCH: GARCH Dynamics per Regime

In [ ]:
import sys
import importlib
from pathlib import Path

if 'PROJECT_ROOT' in globals():
    project_root = Path(PROJECT_ROOT)
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'Seminar QF', cwd.parent]
    project_root = None
    for candidate in candidates:
        if (candidate / 'src').exists():
            project_root = candidate
            break
    if project_root is None:
        tried = '\n'.join(str(c / 'src') for c in candidates)
        raise ModuleNotFoundError(f'Could not locate project root containing src/. Tried:\n{tried}')

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

import src.analysis.regime_analysis as ra
importlib.reload(ra)

ms_garch_metrics = ra.run_ms_garch_analysis(ms_garch, verbose=True)

## MS-GARCH: Alpha, Beta & Persistence Over Time

In [ ]:
ts_median = ms_garch.groupby('date').agg(
    alpha_0=('alpha_0', 'median'),
    alpha_1=('alpha_1', 'median'),
    beta_0=('beta_0',  'median'),
    beta_1=('beta_1',  'median'),
)
ts_median['persistence_0'] = ts_median['alpha_0'] + ts_median['beta_0']
ts_median['persistence_1'] = ts_median['alpha_1'] + ts_median['beta_1']

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(ts_median.index, ts_median['alpha_0'],       color='#3498db', linewidth=2, linestyle='-',  label='α  — Regime 0 (Low Vol)')
ax.plot(ts_median.index, ts_median['beta_0'],        color='#3498db', linewidth=2, linestyle='--', label='β  — Regime 0 (Low Vol)')
ax.plot(ts_median.index, ts_median['persistence_0'], color='#3498db', linewidth=2, linestyle=':',  label='α+β — Regime 0 (Low Vol)')

ax.plot(ts_median.index, ts_median['alpha_1'],       color='#e74c3c', linewidth=2, linestyle='-',  label='α  — Regime 1 (High Vol)')
ax.plot(ts_median.index, ts_median['beta_1'],        color='#e74c3c', linewidth=2, linestyle='--', label='β  — Regime 1 (High Vol)')
ax.plot(ts_median.index, ts_median['persistence_1'], color='#e74c3c', linewidth=2, linestyle=':',  label='α+β — Regime 1 (High Vol)')

ax.axhline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Stationarity bound (α+β = 1)')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Parameter value', fontsize=12)
ax.set_title('MS-GARCH: Median α, β & Persistence Over Time  (all firms)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
available_gvkeys = sorted(ms_garch['gvkey'].unique())
print(f"Total firms available: {len(available_gvkeys)}")
print(f"First 20 gvkeys: {available_gvkeys[:20]}")

selected_gvkey = 241456

if selected_gvkey not in available_gvkeys:
    raise ValueError(f"gvkey {selected_gvkey} not found. Choose from available_gvkeys list above.")

firm_ts = (ms_garch[ms_garch['gvkey'] == selected_gvkey]
           [['date', 'alpha_0', 'beta_0', 'alpha_1', 'beta_1']]
           .sort_values('date')
           .copy())
firm_ts['persistence_0'] = firm_ts['alpha_0'] + firm_ts['beta_0']
firm_ts['persistence_1'] = firm_ts['alpha_1'] + firm_ts['beta_1']

n_obs = len(firm_ts)
date_range = f"{firm_ts['date'].min().strftime('%Y-%m')} → {firm_ts['date'].max().strftime('%Y-%m')}"
print(f"\nSelected firm  : gvkey = {selected_gvkey}")
print(f"Observations   : {n_obs} windows")
print(f"Date range     : {date_range}")
print(f"\nParameter summary:")
print(firm_ts[['alpha_0','beta_0','persistence_0','alpha_1','beta_1','persistence_1']].describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(firm_ts['date'], firm_ts['alpha_0'],       color='#3498db', linestyle='-',  linewidth=2, label='α — Regime 0 (Low Vol)')
ax.plot(firm_ts['date'], firm_ts['alpha_1'],       color='#e74c3c', linestyle='-',  linewidth=2, label='α — Regime 1 (High Vol)')
ax.plot(firm_ts['date'], firm_ts['beta_0'],        color='#3498db', linestyle='--', linewidth=2, label='β — Regime 0 (Low Vol)')
ax.plot(firm_ts['date'], firm_ts['beta_1'],        color='#e74c3c', linestyle='--', linewidth=2, label='β — Regime 1 (High Vol)')
ax.plot(firm_ts['date'], firm_ts['persistence_0'], color='#3498db', linestyle=':',  linewidth=2, label='α+β — Regime 0 (Low Vol)')
ax.plot(firm_ts['date'], firm_ts['persistence_1'], color='#e74c3c', linestyle=':',  linewidth=2, label='α+β — Regime 1 (High Vol)')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='Stationarity bound (α+β = 1)')

ax.set_xlabel('Date')
ax.set_ylabel('Parameter Value')
ax.set_title(f'MS-GARCH: α, β & Persistence Over Time  —  Firm {selected_gvkey}', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Filterable Alpha-Beta Scatter (MS-GARCH)

In [ ]:
import os
import re
from pathlib import Path

SELECT_PERIOD = None
SELECT_YEAR = None
SELECT_LEVERAGE_GROUP = None

EXPORT_SCATTERS = True
EXPORT_DIR = str(DATA_OUTPUT / 'ms_garch_alpha_beta_scatter_by_filters')
MAX_POINTS_PER_PLOT = 7000
EXPORT_USE_FILTERED_DATA = False

PERIOD_RANGES = {
    'Pre-COVID': (None, '2020-02-29'),
    'COVID Shock': ('2020-03-01', '2020-12-31'),
    'Recovery': ('2021-01-01', '2022-12-31'),
    'Post-Recovery': ('2023-01-01', None),
}


def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', str(text).strip().lower()).strip('-')


def _resolve_existing_path(candidates):
    for candidate in candidates:
        path_obj = Path(candidate)
        if path_obj.exists():
            return str(path_obj)
    return None


def make_filtered_scatter(df_plot, title_suffix, save_path=None, max_points=7000, show_plot=True):
    if df_plot.empty:
        return False

    plot_n = min(max_points, len(df_plot))
    plot_df = df_plot.sample(plot_n, random_state=42) if len(df_plot) > plot_n else df_plot

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)

    sns.scatterplot(
        data=plot_df,
        x='alpha_0',
        y='beta_0',
        alpha=0.35,
        color='#4C72B0',
        label='Observations',
        ax=axes[0],
    )
    sns.scatterplot(
        data=plot_df,
        x='alpha_1',
        y='beta_1',
        alpha=0.35,
        color='#4C72B0',
        label='Observations',
        ax=axes[1],
    )

    for ax, panel_title in zip(axes, ['Regime 0', 'Regime 1']):
        ax.plot([0, 1], [1, 0], 'r--', label='alpha + beta = 1')
        ax.set_xlim(0, 0.5)
        ax.set_ylim(0.3, 1.05)
        ax.set_xlabel('Alpha')
        ax.set_ylabel('Beta')
        ax.set_title(f'MS-GARCH {panel_title}: Alpha vs Beta')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best')

    plt.suptitle(f'MS-GARCH Alpha-Beta Scatter ({title_suffix})', y=1.02, fontsize=13)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if show_plot:
        plt.show()
    else:
        plt.close(fig)
    return True


ms_scatter = ms_garch.copy()
ms_scatter['date'] = pd.to_datetime(ms_scatter['date'], errors='coerce')

if 'leverage_tercile' not in ms_scatter.columns:
    merged_path = _resolve_existing_path([
        str(DATA_OUTPUT / 'merged_data_with_merton.csv'),
        'data/output/merged_data_with_merton.csv',
        '../data/output/merged_data_with_merton.csv',
    ])
    if merged_path is None:
        raise FileNotFoundError('Could not locate merged_data_with_merton.csv for leverage mapping.')

    lev_df = pd.read_csv(merged_path, usecols=['gvkey', 'date', 'liabilities_total', 'asset_value', 'mkt_cap'])
    lev_df['date'] = pd.to_datetime(lev_df['date'], errors='coerce')
    lev_df['leverage_ratio'] = np.where(
        lev_df['asset_value'].gt(0),
        lev_df['liabilities_total'] / lev_df['asset_value'],
        np.where(lev_df['mkt_cap'].gt(0), lev_df['liabilities_total'] / lev_df['mkt_cap'], np.nan),
    )
    lev_df = lev_df[['gvkey', 'date', 'leverage_ratio']].dropna(subset=['gvkey', 'date'])

    map_df = lev_df[['gvkey', 'leverage_ratio']].dropna()
    q = min(3, map_df['leverage_ratio'].nunique())
    if q >= 2:
        labels = ['Low Leverage', 'Mid Leverage', 'High Leverage'][:q]
        map_df['leverage_tercile'] = pd.qcut(
            map_df['leverage_ratio'], q=q, labels=labels, duplicates='drop'
        )
    else:
        map_df['leverage_tercile'] = 'All'

    firm_lev = map_df.groupby('gvkey')['leverage_tercile'].first().astype(str)
    ms_scatter['leverage_tercile'] = ms_scatter['gvkey'].map(firm_lev).fillna('Unknown')
else:
    ms_scatter['leverage_tercile'] = ms_scatter['leverage_tercile'].astype(str).fillna('Unknown')

ms_scatter['period'] = 'Unassigned'
for label, (start, end) in PERIOD_RANGES.items():
    mask = pd.Series(True, index=ms_scatter.index)
    if start is not None:
        mask &= ms_scatter['date'] >= pd.Timestamp(start)
    if end is not None:
        mask &= ms_scatter['date'] <= pd.Timestamp(end)
    ms_scatter.loc[mask, 'period'] = label
ms_scatter['year'] = ms_scatter['date'].dt.year.astype('Int64')

ms_scatter = ms_scatter.dropna(subset=['alpha_0', 'beta_0', 'alpha_1', 'beta_1'])
if ms_scatter.empty:
    raise ValueError('No rows remain after preprocessing filters.')

ms_scatter_all = ms_scatter.copy()

if SELECT_PERIOD is not None:
    valid_periods = sorted(ms_scatter['period'].dropna().unique())
    if SELECT_PERIOD not in valid_periods:
        raise ValueError(f"Invalid SELECT_PERIOD={SELECT_PERIOD}. Available: {valid_periods}")
    ms_scatter = ms_scatter[ms_scatter['period'] == SELECT_PERIOD].copy()

if SELECT_YEAR is not None:
    valid_years = sorted([int(y) for y in ms_scatter['year'].dropna().unique()])
    if int(SELECT_YEAR) not in valid_years:
        raise ValueError(f"Invalid SELECT_YEAR={SELECT_YEAR}. Available: {valid_years}")
    ms_scatter = ms_scatter[ms_scatter['year'] == int(SELECT_YEAR)].copy()

if SELECT_LEVERAGE_GROUP is not None:
    valid_lev_groups = sorted(ms_scatter['leverage_tercile'].dropna().unique())
    if SELECT_LEVERAGE_GROUP not in valid_lev_groups:
        raise ValueError(
            f"Invalid SELECT_LEVERAGE_GROUP={SELECT_LEVERAGE_GROUP}. Available: {valid_lev_groups}"
        )
    ms_scatter = ms_scatter[ms_scatter['leverage_tercile'] == SELECT_LEVERAGE_GROUP].copy()

if ms_scatter.empty:
    raise ValueError('No rows remain after applying the selected filters.')

print('MS-GARCH Scatter Filter Audit:')
print(f"  Period: {SELECT_PERIOD if SELECT_PERIOD is not None else 'ALL'}")
print(f"  Year: {SELECT_YEAR if SELECT_YEAR is not None else 'ALL'}")
print(f"  Leverage Group: {SELECT_LEVERAGE_GROUP if SELECT_LEVERAGE_GROUP is not None else 'ALL'}")
print(f"  Preview observations: {len(ms_scatter):,}")
print(f"  Preview unique firms: {ms_scatter['gvkey'].nunique():,}")
print(f"  Preview date range: {ms_scatter['date'].min().date()} to {ms_scatter['date'].max().date()}")
print()
print('Global comparison space (before selectors):')
print(f"  Rows: {len(ms_scatter_all):,}")
print(f"  Firms: {ms_scatter_all['gvkey'].nunique():,}")
print(f"  Periods: {sorted(ms_scatter_all['period'].dropna().unique())}")
print(f"  Years: {sorted([int(y) for y in ms_scatter_all['year'].dropna().unique()])[:8]} ...")
print(f"  Leverage groups: {sorted(ms_scatter_all['leverage_tercile'].dropna().unique())}")

display_title = []
if SELECT_PERIOD is not None:
    display_title.append(f"Period={SELECT_PERIOD}")
if SELECT_YEAR is not None:
    display_title.append(f"Year={SELECT_YEAR}")
if SELECT_LEVERAGE_GROUP is not None:
    display_title.append(f"Leverage={SELECT_LEVERAGE_GROUP}")
title_suffix = ', '.join(display_title) if display_title else 'All data'
make_filtered_scatter(ms_scatter, title_suffix=title_suffix, max_points=MAX_POINTS_PER_PLOT, show_plot=True)

if EXPORT_SCATTERS:
    export_base = ms_scatter.copy() if EXPORT_USE_FILTERED_DATA else ms_scatter_all.copy()

    os.makedirs(EXPORT_DIR, exist_ok=True)
    export_count = 0

    def _export_subset(df_subset, filename, title):
        save_path = os.path.join(EXPORT_DIR, filename)
        return make_filtered_scatter(
            df_subset,
            title_suffix=title,
            save_path=save_path,
            max_points=MAX_POINTS_PER_PLOT,
            show_plot=False,
        )

    for period in sorted(export_base['period'].dropna().unique()):
        d = export_base[export_base['period'] == period]
        if d.empty:
            continue
        if _export_subset(d, f"ms_garch_alpha_beta_period_{slugify(period)}.png", f"Period={period}"):
            export_count += 1

    for year in sorted([int(y) for y in export_base['year'].dropna().unique()]):
        d = export_base[export_base['year'] == year]
        if d.empty:
            continue
        if _export_subset(d, f"ms_garch_alpha_beta_year_{year}.png", f"Year={year}"):
            export_count += 1

    for period in sorted(export_base['period'].dropna().unique()):
        sub = export_base[export_base['period'] == period]
        for lev in sorted(sub['leverage_tercile'].dropna().unique()):
            d = sub[sub['leverage_tercile'] == lev]
            if d.empty:
                continue
            if _export_subset(
                d,
                f"ms_garch_alpha_beta_period_{slugify(period)}_leverage_{slugify(lev)}.png",
                f"Period={period}, Leverage={lev}",
            ):
                export_count += 1

    print(f"\nExported {export_count} scatter plot files to: {EXPORT_DIR}")

In [ ]:
leverage_order = ['Low Leverage', 'Mid Leverage', 'High Leverage']
available_groups = [g for g in leverage_order if g in ms_scatter_all['leverage_tercile'].unique()]
n_groups = len(available_groups)

if n_groups == 0:
    raise ValueError('No leverage groups found in ms_scatter_all.')

MAX_POINTS = 5000

fig, axes = plt.subplots(
    n_groups, 2,
    figsize=(12, 4.2 * n_groups),
    sharex=True, sharey=True,
    constrained_layout=False,
)
if n_groups == 1:
    axes = axes[np.newaxis, :]

for row_idx, lev_group in enumerate(available_groups):
    subset = ms_scatter_all[ms_scatter_all['leverage_tercile'] == lev_group].copy()
    n_obs = len(subset)
    if n_obs > MAX_POINTS:
        subset = subset.sample(MAX_POINTS, random_state=42)

    ax0 = axes[row_idx, 0]
    sns.scatterplot(
        data=subset, x='alpha_0', y='beta_0',
        alpha=0.30, color='#4C72B0', s=12,
        ax=ax0, legend=False,
    )
    ax0.plot([0, 1], [1, 0], 'r--', linewidth=1.5, label=r'$\alpha+\beta=1$')
    ax0.set_xlim(0, 0.5)
    ax0.set_ylim(0.3, 1.05)
    ax0.set_xlabel(r'$\alpha$', fontsize=11)
    ax0.set_ylabel(r'$\beta$', fontsize=11)
    ax0.set_title(f'MS-GARCH Regime 0: {lev_group}', fontsize=11, fontweight='bold')
    ax0.grid(True, alpha=0.25)
    ax0.legend(fontsize=9, loc='upper right')

    ax1 = axes[row_idx, 1]
    sns.scatterplot(
        data=subset, x='alpha_1', y='beta_1',
        alpha=0.30, color='#4C72B0', s=12,
        ax=ax1, legend=False,
    )
    ax1.plot([0, 1], [1, 0], 'r--', linewidth=1.5, label=r'$\alpha+\beta=1$')
    ax1.set_xlim(0, 0.5)
    ax1.set_ylim(0.3, 1.05)
    ax1.set_xlabel(r'$\alpha$', fontsize=11)
    ax1.set_ylabel(r'$\beta$', fontsize=11)
    ax1.set_title(f'MS-GARCH Regime 1: {lev_group}', fontsize=11, fontweight='bold')
    ax1.grid(True, alpha=0.25)
    ax1.legend(fontsize=9, loc='upper right')

fig.suptitle(
    r'Distribution of estimated MS-GARCH parameters $\alpha$ and $\beta$ by Leverage Group',
    fontsize=14, fontweight='bold', y=1.02,
)
fig.tight_layout()

save_dir = DATA_OUTPUT / 'ms_garch_alpha_beta_scatter_by_filters'
os.makedirs(save_dir, exist_ok=True)
save_path = save_dir / 'ms_garch_alpha_beta_by_leverage_group.png'
fig.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'Saved: {save_path}')
plt.show()

In [ ]:
import sys, os
import pandas as pd
import numpy as np

_notebook_dir = os.path.dirname(os.path.abspath('regime_analysis.ipynb'))
_project_root = os.path.abspath(os.path.join(_notebook_dir, '..'))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import importlib
import src.analysis.regime_analysis as _ra_mod
importlib.reload(_ra_mod)
from src.analysis.regime_analysis import compute_param_summary_by_company, display_param_summary

msgarch_path = os.path.join(_project_root, 'data', 'output',
                            'daily_asset_returns_with_msgarch.csv')
df_msgarch = pd.read_csv(msgarch_path)
df_msgarch['date'] = pd.to_datetime(df_msgarch['date'])

windows_path = os.path.join(_project_root, 'data', 'cds_filters',
                            'gvkey_maturity_simulation_windows.csv')
windows_df = pd.read_csv(windows_path)
windows_df['gvkey'] = windows_df['gvkey'].astype(int)

has_nu = 'nu_0' in df_msgarch.columns and 'nu_1' in df_msgarch.columns
print(f"MS-GARCH data : {df_msgarch.shape[0]:,} rows · {df_msgarch['gvkey'].nunique()} companies")
print(f"Nu columns present: {has_nu}")
print(f"Windows file  : {len(windows_df)} entries · maturities: {sorted(windows_df['maturity'].unique())}")
print()

summaries = {}

for MATURITY in ['1Y', '3Y', '5Y']:
    print('=' * 72)
    print(f'  MATURITY: {MATURITY}')
    print('=' * 72)

    save_csv = os.path.join(_project_root, 'data', 'output',
                            f'ms_garch_alpha_beta_nu_summary_{MATURITY}.csv')

    summary_long = compute_param_summary_by_company(
        df_msgarch, windows_df, maturity=MATURITY, save_path=save_csv
    )
    summaries[MATURITY] = summary_long

    print(f"Rows in long-form table : {len(summary_long)}")
    print(f"Pivoted wide view (regime → param → statistic):\n")
    display_param_summary(summary_long)
    print()

print("Done — three CSV files saved:")
for mat in ['1Y', '3Y', '5Y']:
    path = os.path.join(_project_root, 'data', 'output',
                        f'ms_garch_alpha_beta_nu_summary_{mat}.csv')
    print(f"  {path}")

In [ ]:
import shutil
from pathlib import Path

old_dirs = [
    DATA_OUTPUT / 'parameter_scatter_by_period_and_leverage',
    Path('data/output/parameter_scatter_by_period_and_leverage'),
    Path('../data/output/parameter_scatter_by_period_and_leverage'),
    Path('data/output/ms_garch_alpha_beta_scatter_by_filters'),
]

removed = 0
for old_dir in old_dirs:
    if old_dir.resolve() == (DATA_OUTPUT / 'ms_garch_alpha_beta_scatter_by_filters').resolve():
        continue
    if old_dir.exists():
        for p in old_dir.rglob('*'):
            if p.is_file():
                p.unlink()
                removed += 1
        try:
            shutil.rmtree(old_dir)
        except Exception:
            pass

print(f'Removed {removed} legacy/misplaced scatter file(s).')
print(f'Canonical export dir: {DATA_OUTPUT / "ms_garch_alpha_beta_scatter_by_filters"}')
try:
    n_canon = sum(1 for p in (DATA_OUTPUT / 'ms_garch_alpha_beta_scatter_by_filters').rglob('*') if p.is_file())
    print(f'  → {n_canon} files present in canonical dir.')
except Exception:
    print('  → Canonical dir not yet created (run scatter cell first).')